In [1]:
from pathlib import Path
import pandas as pd 
import numpy as np
import pingouin as pg
from dowhy import CausalModel
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

import modules

pd.set_option('display.max_columns', None)

### Data preprocessing (2024 Data)

In [2]:
mode = 'Train' # 'Train' or 'Test/Congestion' or 'Test/Non_Congest'
    
trn_rts_df = modules.data_prep(mode)

res_df = modules.strat_folds(trn_rts_df, n_folds=5, stratify_cols=["day_of_week", "time_of_day"])

print(len(res_df))
res_df.head()


199800


,date_of_trip,day_of_week,time_of_day,PULocationID,DOLocationID,uber_vol,uber_trip_dist,uber_mph,uber_trip_dur,uber_wait_dur,uber_wait_ratio,uber_fare_per_mile,uber_pay_per_mile,lyft_vol,lyft_trip_dist,lyft_mph,lyft_trip_dur,lyft_wait_dur,lyft_wait_ratio,lyft_fare_per_mile,lyft_pay_per_mile,tot_vol,uber_pay,uber_fare,lyft_pay,lyft_fare,uber_wait,lyft_wait,lyft_share,month_of_trip,sunday_date,fold
0,2024-01-01,0,2,7,7,41,0.892927,11.505134,4.982439,5.090488,1.085918,12.954292,4.188248,28,1.192714,11.224988,6.850714,2.287500,0.402386,10.578263,6.784952,69,4.188248,12.954292,6.784952,10.578263,5.090488,2.287500,0.405797,2024-01,2023-12-30,1
1,2024-01-01,0,2,7,129,18,2.632222,12.911368,13.285000,5.641111,0.473235,6.689509,3.362486,19,2.420526,14.729228,10.046316,2.914737,0.315553,5.875484,3.917398,37,3.362486,6.689509,3.917398,5.875484,5.641111,2.914737,0.513514,2024-01,2023-12-30,2
2,2024-01-01,0,2,7,179,16,1.315000,11.499533,7.094375,5.096250,0.787224,9.592964,3.641561,15,1.124600,11.713507,5.872000,1.821333,0.342103,9.307941,6.381631,31,3.641561,9.592964,6.381631,9.307941,5.096250,1.821333,0.483871,2024-01,2023-12-30,0
3,2024-01-01,0,2,7,223,34,1.883529,12.813989,8.185882,5.561471,0.833741,8.992748,2.841334,16,1.331125,10.645015,7.882500,2.163750,0.301799,8.697696,5.717971,50,2.841334,8.992748,5.717971,8.697696,5.561471,2.163750,0.320000,2024-01,2023-12-30,2
4,2024-01-01,0,2,7,265,14,23.998571,34.367627,40.161429,7.330714,0.194777,5.403085,2.305692,12,20.411333,35.421365,34.518333,3.275000,0.100611,4.288168,2.389737,26,2.305692,5.403085,2.389737,4.288168,7.330714,3.275000,0.461538,2024-01,2023-12-30,1


### Grid Search (Hyperparameters)

In [3]:
from itertools import product

learn_rate = [0.03, 0.04, 0.05]
max_depth = [4, 5, 6]
estimators = [800, 900, 1000]

lshare_res = pd.DataFrame(columns = ['Learn Rate', 'Max Depth', 'N Estimators', 'Test RMSE'])

for lr, md, n_est in product(learn_rate, max_depth, estimators):
    
    res_df, lshare_report, lshare_model = modules.xgb_coeffs(
        df=res_df,
        fold_col="fold",
        objective='reg:squarederror',
        predictor_cols=["uber_wait", "lyft_wait"],
        target_col="lyft_share",
        learning_rate=lr,
        max_depth=md,
        n_estimators=n_est,
    )
    
    lshare_res.loc[len(lshare_res)] = [lr, md, n_est, lshare_report["rmse_test"].mean()]

lfare_res = pd.DataFrame(columns = ['Learn Rate', 'Max Depth', 'N Estimators', 'Test RMSE'])

for lr, md, n_est in product(learn_rate, max_depth, estimators):
    
    res_df, lfare_report, lfare_model = modules.xgb_coeffs(
        df=res_df,
        fold_col="fold",
        objective='reg:squarederror',
        predictor_cols=["uber_wait", "lyft_wait"],
        target_col="lyft_fare",
        learning_rate=lr,
        max_depth=md,
        n_estimators=n_est,
    )
    
    lfare_res.loc[len(lfare_res)] = [lr, md, n_est, lfare_report["rmse_test"].mean()]

ufare_res = pd.DataFrame(columns = ['Learn Rate', 'Max Depth', 'N Estimators', 'Test RMSE'])

for lr, md, n_est in product(learn_rate, max_depth, estimators):
    
    res_df, ufare_report, ufare_model = modules.xgb_coeffs(
        df=res_df,
        fold_col="fold",
        objective='reg:squarederror',
        predictor_cols=["uber_wait", "lyft_wait"],
        target_col="uber_fare",
        learning_rate=lr,
        max_depth=md,
        n_estimators=n_est,
    )
    
    ufare_res.loc[len(ufare_res)] = [lr, md, n_est, ufare_report["rmse_test"].mean()]


In [4]:
ufare_res.sort_values('Test RMSE', ascending=True).head()

,Learn Rate,Max Depth,N Estimators,Test RMSE
0,0.03,4.0,800.0,4.426083
1,0.03,4.0,900.0,4.427126
2,0.03,4.0,1000.0,4.428023
9,0.04,4.0,800.0,4.429484
10,0.04,4.0,900.0,4.430991


In [5]:
lfare_res.sort_values('Test RMSE', ascending=True).head()

,Learn Rate,Max Depth,N Estimators,Test RMSE
0,0.03,4.0,800.0,3.327538
1,0.03,4.0,900.0,3.328354
2,0.03,4.0,1000.0,3.329263
9,0.04,4.0,800.0,3.329847
10,0.04,4.0,900.0,3.330984


In [6]:
lshare_res.sort_values('Test RMSE', ascending=True).head()

,Learn Rate,Max Depth,N Estimators,Test RMSE
0,0.03,4.0,800.0,0.082179
1,0.03,4.0,900.0,0.082195
2,0.03,4.0,1000.0,0.082211
9,0.04,4.0,800.0,0.082219
10,0.04,4.0,900.0,0.082239


In [7]:
ufare_res = ufare_res.sort_values('Test RMSE', ascending=True).reset_index(drop=True)
lfare_res = lfare_res.sort_values('Test RMSE', ascending=True).reset_index(drop=True)
lshare_res = lshare_res.sort_values('Test RMSE', ascending=True).reset_index(drop=True)

u_lr = float(ufare_res.loc[0, 'Learn Rate'])
u_md = int(ufare_res.loc[0, 'Max Depth'])
u_ne = int(ufare_res.loc[0, 'N Estimators'])

l_lr = float(lfare_res.loc[0, 'Learn Rate'])
l_md = int(lfare_res.loc[0, 'Max Depth'])
l_ne = int(lfare_res.loc[0, 'N Estimators'])

s_lr = float(lshare_res.loc[0, 'Learn Rate'])
s_md = int(lshare_res.loc[0, 'Max Depth'])
s_ne = int(lshare_res.loc[0, 'N Estimators'])

print(f"Best Hyperparameters for Uber Fare: Learning Rate = {u_lr}, Max Depth = {u_md}, N Estimators = {u_ne}")
print(f"Best Hyperparameters for Lyft Fare: Learning Rate = {l_lr}, Max Depth = {l_md}, N Estimators = {l_ne}")
print(f"Best Hyperparameters for Lyft Share: Learning Rate = {s_lr}, Max Depth = {s_md}, N Estimators = {s_ne}")

Best Hyperparameters for Uber Fare: Learning Rate = 0.03, Max Depth = 4, N Estimators = 800
Best Hyperparameters for Lyft Fare: Learning Rate = 0.03, Max Depth = 4, N Estimators = 800
Best Hyperparameters for Lyft Share: Learning Rate = 0.03, Max Depth = 4, N Estimators = 800


### Nuisance Function Models (Cross Validation)

In [8]:
res_df, ufare_report, ufare_model = modules.xgb_coeffs(
    df=res_df,
    fold_col="fold",
    objective='reg:squarederror',
    predictor_cols=["uber_wait", "lyft_wait"],
    target_col="uber_fare",
    learning_rate=u_lr,
    max_depth=u_md,
    n_estimators=u_ne,
)

res_df, lfare_report, lfare_model = modules.xgb_coeffs(
    df=res_df,
    fold_col="fold",
    objective='reg:squarederror',
    predictor_cols=["uber_wait", "lyft_wait"],
    target_col="lyft_fare",
    learning_rate=l_lr,
    max_depth=l_md,
    n_estimators=l_ne,
)

res_df, lshare_report, lshare_model = modules.xgb_coeffs(
    df=res_df,
    fold_col="fold",
    objective='reg:squarederror',
    predictor_cols=["uber_wait", "lyft_wait"],
    target_col="lyft_share",
    learning_rate=s_lr,
    max_depth=s_md,
    n_estimators=s_ne,
)

### Orthogonal Regression Model

In [9]:
results = modules.fit_linear_regression(
    df=res_df,
    predictor_cols=['uber_fare_error', 'lyft_fare_error'],
    target_col='lyft_share_error',
    intercept=False
)

print(results.summary())


                                 OLS Regression Results                                
Dep. Variable:       lyft_share_error   R-squared (uncentered):                   0.056
Model:                            OLS   Adj. R-squared (uncentered):              0.056
Method:                 Least Squares   F-statistic:                              5914.
Date:                Thu, 05 Mar 2026   Prob (F-statistic):                        0.00
Time:                        03:52:29   Log-Likelihood:                      2.2151e+05
No. Observations:              199800   AIC:                                 -4.430e+05
Df Residuals:                  199798   BIC:                                 -4.430e+05
Df Model:                           2                                                  
Covariance Type:            nonrobust                                                  
                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------